In [1]:
import json, io
import numpy as np
import torch
import torch.nn as nn
import chess
import re
import random
import pickle
import onnxruntime as ogpu

# configuring device
try:
    device = xm.xla_device()
    print("Running on the TPU")
except:
    if torch.cuda.is_available():
        device = torch.device('cuda:0')
        print('Running on the GPU')
        torch.cuda.synchronize()
    else:
        device = torch.device('cpu')
        print('Running on the CPU')

Running on the GPU


In [2]:
from helperfuncs import *

LONG_INTERVAL = 200
SHORT_INTERVAL = 10

EVAL_SET_SIZE = 65536


# Total (rough, assuming equal training time):
# Easy puzzles: 22.4%, Mid puzzles: 19.3%, Hard puzzles: 12.0%, Mates: 7.8%, Openings: 10.9%, Midgames: 14.1%, Endgames: 13.5%

EPOCH_SIZE = 65536
BATCH_SIZE = 4096
VAL_SIZE = 65536 * 12

learning_rate = 0.001
l2r = 0
checkpoint = 0
save = True
model_name = "parakeet_1"

TIME_CHECK = True

In [3]:
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()

        hidden = max(channels // reduction, 8)

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.fc = nn.Sequential(
            nn.Linear(channels, hidden),
            nn.SiLU(),
            nn.Linear(hidden, channels),
            nn.Sigmoid(),
        )

    def forward(self, x):
        # x: (B, C, 8, 8)

        b, c, _, _ = x.shape

        scale = self.pool(x).view(b, c)
        scale = self.fc(scale).view(b, c, 1, 1)

        return x * scale


class ResidualBlock(nn.Module):
    def __init__(self, channels=128, use_se=True):
        super().__init__()

        self.conv1 = nn.Conv2d(
            channels,
            channels,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.bn1 = nn.BatchNorm2d(channels)
        self.act1 = nn.SiLU()

        self.conv2 = nn.Conv2d(
            channels,
            channels,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.bn2 = nn.BatchNorm2d(channels)

        self.se = SEBlock(channels) if use_se else nn.Identity()

        self.act2 = nn.SiLU()

    def forward(self, x):
        residual = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.act1(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out = self.se(out)

        out = out + residual
        out = self.act2(out)

        return out


class ChessEvaluationNet(nn.Module):
    def __init__(
        self,
        channels=128,
        num_blocks=8,
        value_channels=32,
        hidden_size=256,
        use_se=True,
    ):
        super().__init__()

        # Input:
        # (B, 1, 8, 8)
        #
        # Output:
        # (B, 1), in [0, 1]

        # --------------------------------------------------
        # Stem
        # --------------------------------------------------

        self.stem = nn.Sequential(
            nn.Conv2d(
                1,
                channels,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(channels),
            nn.SiLU(),
        )

        # --------------------------------------------------
        # Residual tower
        # --------------------------------------------------

        self.residual_tower = nn.Sequential(
            *[
                ResidualBlock(
                    channels=channels,
                    use_se=use_se,
                )
                for _ in range(num_blocks)
            ]
        )

        # --------------------------------------------------
        # Value head
        # --------------------------------------------------

        self.value_head = nn.Sequential(
            nn.Conv2d(
                channels,
                value_channels,
                kernel_size=1,
                bias=False,
            ),
            nn.BatchNorm2d(value_channels),
            nn.SiLU(),

            nn.Flatten(),

            nn.Linear(
                value_channels * 8 * 8,
                hidden_size,
            ),
            nn.SiLU(),

            nn.Linear(hidden_size, 1),

            # Evaluation between 0 and 1
            nn.Sigmoid(),
        )

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters())

    def forward(self, x):
        x = self.stem(x)
        x = self.residual_tower(x)
        x = self.value_head(x)

        return x

In [4]:
m = ChessEvaluationNet().to(device)
print(m.count_parameters())
m(torch.rand(2, 1, 8, 8, device=device))

2911233


tensor([[0.4433],
        [0.5080]], device='cuda:0', grad_fn=<SigmoidBackward0>)

In [ ]:


print("Dataset info:")
print(f"- Drawish positions: 65536 * {NDP} = {65536 * NDP}")
print(f"- Advantageous positions: 65536 * {NAP} = {65536 * NAP}")
print(f"- Winning positions: 65536 * {NWP} = {65536 * NWP}")
print(f"- Total = {65536 * (NAP + NDP + NWP)}")

print("----------------")


DRAW_VALIDATION_SET = [25, 200, 300, 525]
ADV_VALIDATION_SET = [75, 200, 325, 450]
WIN_VALIDATION_SET = [50, 100, 225, 350]

# Build validation sets
print("Target validation set size:", VAL_SIZE)

val_draw_boards, val_draw_features, val_draw_evals = [], [], []

for dvs in DRAW_VALIDATION_SET:
    vb, vf, ve = read_eval_data("draw", dvs)
    val_draw_boards += vb
    val_draw_features += vf
    val_draw_evals += ve
print("Draws board validation set dimensions: ", np.shape(val_draw_boards))

val_adv_boards, val_adv_features, val_adv_evals = [], [], []

for avs in ADV_VALIDATION_SET:
    vb, vf, ve = read_eval_data("adv", avs)
    val_adv_boards += vb
    val_adv_features += vf
    val_adv_evals += ve
print("Advs board validation set dimensions: ", np.shape(val_adv_boards))

val_win_boards, val_win_features, val_win_evals = [], [], []

for wvs in WIN_VALIDATION_SET:
    vb, vf, ve = read_eval_data("winning", wvs)
    val_win_boards += vb
    val_win_features += vf
    val_win_evals += ve
print("Win board validation set dimensions: ", np.shape(val_win_boards))


val_position_list = list(val_draw_boards) + list(val_adv_boards) + list(val_win_boards)
val_features_list = list(val_draw_features) + list(val_adv_features) + list(val_win_features)
val_eval_list = list(val_draw_evals) + list(val_adv_evals) + list(val_win_evals)

print("Total board validation set dimensions: ", np.shape(val_position_list))


Dataset info:
- Drawish positions: 65536 * 528 = 34603008
- Advantageous positions: 65536 * 462 = 30277632
- Winning positions: 65536 * 396 = 25952256
- Total = 90832896
----------------
Target validation set size: 786432
Draws board validation set dimensions:  (262144, 1, 8, 8)
Advs board validation set dimensions:  (262144, 1, 8, 8)
Win board validation set dimensions:  (262144, 1, 8, 8)
Total board validation set dimensions:  (786432, 1, 8, 8)


In [6]:
# Training data loader

class DataLoader:

    def __init__(self, dataset):
        self.dataset = dataset
        self.pointer = 0
        self.length = 65536
        self.data = None

    def get_dataset(self):

        done=False

        if self.dataset == position.DRAW:
            ri = random.randint(0, NDP)
            while ri in DRAW_VALIDATION_SET:
                ri = random.randint(0, NDP)
            while not done:
                try:
                    self.data = read_eval_data("draw", ri)
                    done = True
                except:
                    print("Could not read draw", ri)
                ri = random.randint(0, NDP)
        elif self.dataset == position.ADVANTAGE:
            ri = random.randint(0, NAP)
            while ri in ADV_VALIDATION_SET:
                ri = random.randint(0, NAP)
            while not done:
                try:
                    self.data = read_eval_data("adv", ri)
                    done = True
                except:
                    print("Could not read adv", ri)
        elif self.dataset == position.WINNING:
            ri = random.randint(0, NWP)
            while ri in WIN_VALIDATION_SET:
                ri = random.randint(0, NWP)
            while not done:
                try:
                    self.data = read_eval_data("winning", ri)
                    done = True
                except:
                    print("Could not read winning", ri)

    def get_data(self, num):
        res_pos, res_eval = [], []
        if self.data is None:
            self.get_dataset()
        while num > 0:
            res_pos += self.data[0][self.pointer : num + self.pointer]
            res_eval += self.data[2][self.pointer : num + self.pointer]
            if num >= self.length - self.pointer:
                num -= (self.length - self.pointer)
                self.pointer = 0
                self.get_dataset()
            else:
                self.pointer += num
                num = 0
        return res_pos, res_eval


In [ ]:
import time
import random
import os
import math
import statistics

model = ChessEvaluationNet().to(device=device)

checkpoint = -1
learning_rate = 0.0003
l2r = 0
tvl = 0

from pytorch_optimizer import SOAP
optimizer = SOAP(model.parameters(), lr=learning_rate)
loss_fn = nn.MSELoss()


if checkpoint == 0:
    num_epoch = 0
    best_vloss = 1000
    model.train()
    with open(f"D:/parakeet/models/{model_name}_loss.csv", "w") as file:
        file.write("")
    print("Using new models.")

elif checkpoint == -1:
    print("Loading from last checkpoint.")
    checkpoint_file = torch.load(f"D:/parakeet/models/{model_name}.pickle", weights_only=True, map_location=device)
    model.load_state_dict(checkpoint_file["model_state_dict"])
    model.to(device=device)
    optimizer.load_state_dict(checkpoint_file["optimizer_state_dict"])

    # Update learning rate in case it is changed midway.
    for g in optimizer.param_groups:
      g['lr'] = learning_rate

    num_epoch = checkpoint_file["epoch"] + 1
    best_vloss = checkpoint_file["best_loss"]
    model.train()
    print(f"Best validation loss at checkpoint: {best_vloss}")

elif checkpoint == -2:
    print("Switching to new dataset.")
    checkpoint_file = torch.load(f"D:/parakeet/models/{model_name}.pickle", weights_only=True, map_location=device)
    model.load_state_dict(checkpoint_file["model_state_dict"])
    model.to(device=device)

    # Update learning rate in case it is changed midway.
    for g in optimizer.param_groups:
      g['lr'] = learning_rate

    num_epoch = 0
    best_vloss = 1000
    model.train()


else:
    print(f"Loading from epoch {checkpoint}.")
    checkpoint_file = torch.load(f"D:/parakeet/models/{model_name}_{checkpoint}.pickle", weights_only=True, map_location=device)
    model.load_state_dict(checkpoint_file["model_state_dict"])
    model.to(device=device)
    optimizer.load_state_dict(checkpoint_file["optimizer_state_dict"])

    # Update learning rate in case it is changed midway.
    for g in optimizer.param_groups:
      g['lr'] = learning_rate

    num_epoch = checkpoint_file["epoch"] + 1
    best_vloss = checkpoint_file["best_loss"]
    model.train()
    print(f"Best validation loss at checkpoint: {best_vloss}")



print(model)
print("Batch size = ", BATCH_SIZE)
print("Validation size = ", VAL_SIZE)
print("L2 regularisation strength =", l2r)
print("Learning rate =", learning_rate)

def warmup_then_expo(epoch):
  if epoch < 58:
    return epoch / 58
  else:
    return (0.999 ** (epoch - 58))
#scheduler = lr_scheduler.LambdaLR(optimizer, lr_lambda=warmup_then_expo)

tlr, tl, vl = [], [], []

running_mean = 0
M2 = 0
readings = 0

# Setup dataloaders
draw_loader = DataLoader(position.DRAW)
adv_loader = DataLoader(position.ADVANTAGE)
win_loader = DataLoader(position.WINNING)

for epoch in range(num_epoch, 800000):

    bl, el = [], []
    bl3, el3, bl4, el4, bl5, el5 = [], [], [], [], [], []
    start = time.time()

    bl3, el3 = draw_loader.get_data(round(EPOCH_SIZE * 0.4))
    bl4, el4 = adv_loader.get_data(round(EPOCH_SIZE * 0.3))
    bl5, el5 = win_loader.get_data(round(EPOCH_SIZE * 0.3))

    bl = (bl3 + bl4 + bl5)
    el = (el3 + el4 + el5)
    #print("Shape of training set boards:", np.shape(bl))

    bl = np.asarray(bl, dtype=np.float32)
    el = np.asarray(el, dtype=np.float32)

    perm = np.random.permutation(len(bl))

    bl = bl[perm]
    el = el[perm]

    cl = 0
    clr = 0
    norms = []
    data_load_time = time.time() - start
    readings += 1
    delta = data_load_time - running_mean
    running_mean += delta / readings
    delta2 = data_load_time - running_mean
    M2 += (delta * delta2)
    variance = M2 / readings
    print(f"Data loaded in {data_load_time} seconds. Mean {running_mean}, Stdev {variance ** 0.5}")
    if (len(bl) != EPOCH_SIZE) or (len(el) != EPOCH_SIZE):
        print("Dataset is invalid.")
        continue
    else:
        start = time.time()
        for batch in range(EPOCH_SIZE // BATCH_SIZE):
            tb = torch.tensor(bl[batch * BATCH_SIZE : (batch + 1) * BATCH_SIZE], device=device, dtype=torch.float).reshape(BATCH_SIZE, 1, 8, 8)
            te = torch.tensor(el[batch * BATCH_SIZE : (batch + 1) * BATCH_SIZE], device=device, dtype=torch.float).reshape(BATCH_SIZE, 1)

            optimizer.zero_grad()
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                out = model(tb)
                loss = loss_fn(out, te)
                l = loss.item()
                cl += l
                if l2r != 0:
                    l2 = sum(p.pow(2).sum() for p in model.parameters())
                    loss += l2 * l2r
                    llr = loss.item()
                    clr += llr
            loss.backward()


            optimizer.step()
            tl.append(l)
            if l2r != 0:
                tlr.append(llr)
            completion = int(20 * batch / (EPOCH_SIZE // BATCH_SIZE)) + 1
            if l2r != 0:
                print("\r" + f"[{'-' * completion} {' ' * (20 - completion)}]     Loss {round(l, 6)}, Regularised loss {round(llr, 6)}", end = "")
            else:
                print("\r" + f"[{'-' * completion} {' ' * (20 - completion)}]     Loss {round(l, 6)}", end = "")
        if l2r != 0: print(f"\nEpoch {epoch}, loss {round(cl / (EPOCH_SIZE // BATCH_SIZE), 6)}, regularised loss {round(clr / (EPOCH_SIZE // BATCH_SIZE), 6)}, completed in {time.time() - start} seconds.")
        else: print(f"\nEpoch {epoch}, loss {round(cl / (EPOCH_SIZE // BATCH_SIZE), 6)}, completed in {time.time() - start} seconds.")

        if (epoch % SHORT_INTERVAL == 0):
            with torch.inference_mode():
                model.eval()
                tvl = 0
                start = time.time()
                for vbatch in range(VAL_SIZE // BATCH_SIZE):
                    vp = torch.tensor(val_position_list[vbatch * BATCH_SIZE : (vbatch + 1) * BATCH_SIZE], device=device, dtype=torch.float).reshape(BATCH_SIZE, 1, 8, 8)
                    ve = torch.tensor(val_eval_list[vbatch * BATCH_SIZE : (vbatch + 1) * BATCH_SIZE], device=device, dtype=torch.float).reshape(BATCH_SIZE, 1)
                    out = model(vp)
                    loss = loss_fn(out, ve)
                    tvl += loss.item()

                print("Validation loss", round(tvl / (VAL_SIZE // BATCH_SIZE), 6), "completed in", time.time() - start, "seconds.")
                vl.append(tvl / (VAL_SIZE // BATCH_SIZE))
                if (tvl / (VAL_SIZE // BATCH_SIZE)) < best_vloss:
                    print("New best model!")
                    best_vloss = tvl / (VAL_SIZE // BATCH_SIZE)
                    torch.save(model.state_dict(), f"D:/parakeet/models/best_{model_name}.pickle")

                model.train()

        #before_lr = learning_rate
        #learning_rate *= 2
        #for g in optimizer.param_groups:
        #    g['lr'] = learning_rate
        #print(f"Epoch {epoch} : lr {before_lr} -> {learning_rate}")


        if save:
            if epoch % LONG_INTERVAL == 0:
                torch.save({"epoch": epoch, "model_state_dict": model.state_dict(), "optimizer_state_dict": optimizer.state_dict(), "best_loss": best_vloss}, f"D:/parakeet/models/{model_name}_{epoch}.pickle")
            if epoch % SHORT_INTERVAL == 0:
                torch.save({"epoch": epoch, "model_state_dict": model.state_dict(), "optimizer_state_dict": optimizer.state_dict(), "best_loss": best_vloss}, f"D:/parakeet/models/{model_name}.pickle")
            try:
                os.remove(f"D:/parakeet/models/{model_name}_{epoch - LONG_INTERVAL}.pickle")
            except:
                pass

        with open(f"D:/parakeet/models/{model_name}_loss.csv", "a") as file:
            file.write(f"{epoch}, {round(cl / (EPOCH_SIZE // BATCH_SIZE), 6)}, {round(tvl / (VAL_SIZE // BATCH_SIZE), 6)}\n")


Loading from last checkpoint.
Best validation loss at checkpoint: 0.01427042381207381
ChessEvaluationNet(
  (stem): Sequential(
    (0): Conv2d(1, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): SiLU()
  )
  (residual_tower): Sequential(
    (0): ResidualBlock(
      (conv1): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (act1): SiLU()
      (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (se): SEBlock(
        (pool): AdaptiveAvgPool2d(output_size=1)
        (fc): Sequential(
          (0): Linear(in_features=128, out_features=8, bias=True)
          (1): SiLU()
   

In [5]:
import torch.onnx

model = ChessEvaluationNet()

checkpoint_file = torch.load(r"C:\Projects\Parakeet\best_parakeet_1.pickle", weights_only=True, map_location=device)
print(checkpoint_file)
model.load_state_dict(checkpoint_file)
model.to(device=device)
model.eval()

# Input to the model
x = torch.randn(64, 1, 8, 8, requires_grad=True, device=device)
torch_out = model(x)

# Export the model
#torch.onnx.export(model,               # model being run
#                  x,                         # model input (or a tuple for multiple inputs)
#                  "C:/Projects/Parakeet/best_parakeet_1.onnx",   # where to save the model (can be a file or file-like object)
#                  export_params=True,        # store the trained parameter weights inside the model file
#                  opset_version=14,          # the ONNX version to export the model to
#                  do_constant_folding=True,  # whether to execute constant folding for optimization
#                  input_names = ['input'],   # the model's input names
#                  output_names = ['output'], # the model's output names
#                  dynamic_axes={'input' : {0 : 'batch_size'},    # variable length axes
#
#
#              'output' : {0 : 'batch_size'}})
#

OrderedDict([('stem.0.weight', tensor([[[[ 1.6039e-02, -1.5640e-01,  9.2777e-02],
          [-1.0955e-01,  1.8360e-01,  1.4707e-01],
          [ 1.3497e-01,  2.7736e-01,  7.6803e-02]]],


        [[[-7.3718e-02, -6.4567e-02, -8.8228e-02],
          [-2.2972e-01,  3.6812e-01, -1.3150e-01],
          [-1.4605e-01,  6.9073e-03, -7.0444e-02]]],


        [[[ 1.0535e-01,  1.9783e-01, -1.1575e-01],
          [ 2.6071e-02,  1.5462e-01,  1.2602e-01],
          [-1.1308e-02, -1.5871e-01, -2.3939e-01]]],


        ...,


        [[[-2.3907e-02, -1.4277e-02, -2.3987e-01],
          [-4.6065e-05,  1.9981e-01,  5.8114e-02],
          [-5.4138e-03, -1.0366e-01,  2.0029e-02]]],


        [[[ 6.5753e-02,  1.3654e-01,  5.6016e-02],
          [ 9.9913e-02,  3.1255e-01, -6.1454e-02],
          [-6.2777e-02, -9.8269e-02, -1.2159e-01]]],


        [[[ 1.1252e-01, -2.6708e-01,  3.6938e-02],
          [-2.4687e-01, -2.2049e-01, -1.0099e-01],
          [-6.2832e-02, -2.1020e-01, -3.7276e-02]]]], device='cuda:

In [ ]:
import chess
import torch


def evaluate_next_moves(model, device, board):
    """
    Evaluate every legal move from the current board position.
    Returns moves sorted from highest network evaluation to lowest.
    """
    model.eval()
    results = []

    with torch.inference_mode():
        for move in board.legal_moves:
            child = board.copy()
            child.push(move)

            # EXACT same representation used when creating training data
            board_map = fast_board_to_boardmap(child)

            x = torch.tensor(
                board_map,
                dtype=torch.float32,
                device=device,
            ).unsqueeze(0)

            # fast_board_to_boardmap: (1, 8, 8)
            # after unsqueeze:      (1, 1, 8, 8)

            raw_eval = model(x).item()

            results.append(
                (move, board.san(move), raw_eval)
            )

    results.sort(key=lambda x: x[2], reverse=True)

    print(f"\nPosition after: {board.fen()}")
    print(f"Side to move: {'White' if board.turn == chess.WHITE else 'Black'}")
    print("-" * 48)

    for move, san, value in results:
        print(f"{move.uci():6s}  {san:8s}  {value:.6f}")

    model.train()

    return results


def chess_cli(model, device):
    board = chess.Board()

    print("Chess evaluation CLI")
    print("--------------------")
    print("Enter moves in UCI format, e.g. d2d4, g8f6, e7e8q")
    print("Commands:")
    print("  board  - show the current board")
    print("  fen    - show the current FEN")
    print("  undo   - undo the previous move")
    print("  reset  - reset to the starting position")
    print("  eval   - evaluate current position's legal moves")
    print("  quit   - exit")
    print()

    # Show evaluations from the starting position immediately
    evaluate_next_moves(model, device, board)

    while True:
        try:
            user_input = input("\n> ").strip().lower()
        except (EOFError, KeyboardInterrupt):
            print("\nExiting.")
            break

        if not user_input:
            continue

        if user_input in {"quit", "exit", "q"}:
            break

        if user_input == "board":
            print()
            print(board)
            continue

        if user_input == "fen":
            print(board.fen())
            continue

        if user_input == "reset":
            board.reset()
            print("\nBoard reset.")
            print(board)
            evaluate_next_moves(model, device, board)
            continue

        if user_input == "undo":
            if board.move_stack:
                undone = board.pop()
                print(f"\nUndid {undone.uci()}")
                print(board)
                evaluate_next_moves(model, device, board)
            else:
                print("No moves to undo.")
            continue

        if user_input == "eval":
            evaluate_next_moves(model, device, board)
            continue

        # Treat everything else as a UCI move
        try:
            move = chess.Move.from_uci(user_input)
        except ValueError:
            print(f"Invalid UCI move: {user_input}")
            continue

        if move not in board.legal_moves:
            print(f"Illegal move: {user_input}")
            continue

        san = board.san(move)
        board.push(move)

        print(f"\nPlayed: {user_input} ({san})")
        print(board)

        # Evaluate all possible next moves
        if board.is_game_over():
            print(f"\nGame over: {board.result()}")
        else:
            evaluate_next_moves(model, device, board)


# Start the CLI
chess_cli(model, device)

Chess evaluation CLI
--------------------
Enter moves in UCI format, e.g. d2d4, g8f6, e7e8q
Commands:
  board  - show the current board
  fen    - show the current FEN
  undo   - undo the previous move
  reset  - reset to the starting position
  eval   - evaluate current position's legal moves
  quit   - exit


Position after: rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1
Side to move: White
------------------------------------------------
d2d4    d4        0.522132
g1f3    Nf3       0.519819
e2e4    e4        0.512114
c2c4    c4        0.510700
b1c3    Nc3       0.501794
e2e3    e3        0.499789
g2g3    g3        0.499105
c2c3    c3        0.498617
b2b3    b3        0.495463
a2a3    a3        0.495179
d2d3    d3        0.493877
a2a4    a4        0.492010
b2b4    b4        0.487988
h2h3    h3        0.487764
g1h3    Nh3       0.478092
f2f4    f4        0.476992
h2h4    h4        0.476740
b1a3    Na3       0.476018
f2f3    f3        0.461212
g2g4    g4        0.460254


In [ ]:
# Verify ONNX export against PyTorch on every legal move from the starting position
# Uses fast_board_to_boardmap(), i.e. the exact board representation used to generate the training data.

import numpy as np
import chess
import torch
import onnxruntime as ort

ONNX_PATH = r"C:/Projects/Parakeet/best_parakeet_1.onnx"


def make_starting_move_batch():
    """Return all 20 legal child positions after White's first move.

    x_np has shape (20, 1, 8, 8) and uses the exact training-data encoder.
    """
    root = chess.Board()
    rows = []

    for move in root.legal_moves:
        san = root.san(move)
        child = root.copy(stack=False)
        child.push(move)

        # fast_board_to_boardmap(child) already has shape (1, 8, 8)
        board_map = np.asarray(fast_board_to_boardmap(child), dtype=np.float32)
        rows.append((san, move.uci(), board_map))

    x_np = np.stack([r[2] for r in rows], axis=0)
    return rows, x_np


# ----- Make one identical batch for both backends -----
move_rows, x_np = make_starting_move_batch()
assert x_np.shape == (20, 1, 8, 8), x_np.shape


# ----- PyTorch -----
model.eval()
with torch.inference_mode():
    x_torch = torch.from_numpy(x_np).to(device)
    torch_values = model(x_torch).detach().cpu().numpy().reshape(-1)


# ----- ONNX Runtime -----
available = ort.get_available_providers()
print("ONNX Runtime available providers:", available)

# Prefer CUDA if it is actually available, otherwise use CPU.
providers = ["CUDAExecutionProvider", "CPUExecutionProvider"] \
    if "CUDAExecutionProvider" in available else ["CPUExecutionProvider"]

session = ort.InferenceSession(ONNX_PATH, providers=providers)
print("ONNX Runtime session providers:", session.get_providers())
print("ONNX input:", session.get_inputs()[0].name, session.get_inputs()[0].shape, session.get_inputs()[0].type)
print("ONNX output:", session.get_outputs()[0].name, session.get_outputs()[0].shape, session.get_outputs()[0].type)

input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name
onnx_values = session.run([output_name], {input_name: x_np})[0].reshape(-1)


# ----- Direct numerical comparison -----
abs_diff = np.abs(torch_values - onnx_values)

print("\nPyTorch vs ONNX numerical comparison")
print("-------------------------------------")
print(f"Max absolute difference : {abs_diff.max():.10f}")
print(f"Mean absolute difference: {abs_diff.mean():.10f}")
print(f"RMSE                    : {np.sqrt(np.mean((torch_values - onnx_values) ** 2)):.10f}")

# FP32 ONNX and PyTorch should normally be extremely close.
# This is deliberately a warning rather than an assertion because different
# GPU kernels can introduce small numerical differences.
if abs_diff.max() > 1e-4:
    print("WARNING: PyTorch and ONNX differ by more than 1e-4.")
else:
    print("PASS: PyTorch and ONNX outputs closely agree.")


# ----- Compare each starting move -----
comparison = []
for (san, uci, _), pt, ox in zip(move_rows, torch_values, onnx_values):
    comparison.append((san, uci, float(pt), float(ox), float(abs(pt - ox))))

# The training targets are White-perspective scores, so after a White move,
# larger values are better for White. No 1-value inversion is applied here.
comparison.sort(key=lambda r: r[2], reverse=True)

print("\nStarting position: PyTorch vs ONNX")
print("----------------------------------")
print(f"{'Move':<7} {'UCI':<6} {'PyTorch':>12} {'ONNX':>12} {'|diff|':>12}")
print("-" * 55)
for san, uci, pt, ox, diff in comparison:
    print(f"{san:<7} {uci:<6} {pt:>12.6f} {ox:>12.6f} {diff:>12.8f}")


# ----- Compare rankings independently -----
pt_ranking = sorted(comparison, key=lambda r: r[2], reverse=True)
ox_ranking = sorted(comparison, key=lambda r: r[3], reverse=True)

print("\nPyTorch ranking:")
print(" > ".join(r[0] for r in pt_ranking))

print("\nONNX ranking:")
print(" > ".join(r[0] for r in ox_ranking))

if [r[1] for r in pt_ranking] == [r[1] for r in ox_ranking]:
    print("\nPASS: Full PyTorch and ONNX move rankings are identical.")
else:
    print("\nNOTE: Rankings differ. Check the numerical differences above.")


# ----- Optional second test: random tensors -----
# This catches export errors that might not happen to show up in the 20 opening positions.
rng = np.random.default_rng(12345)
random_np = rng.random((64, 1, 8, 8), dtype=np.float32)

with torch.inference_mode():
    random_pt = model(torch.from_numpy(random_np).to(device)).cpu().numpy()

random_ox = session.run([output_name], {input_name: random_np})[0]
random_diff = np.abs(random_pt - random_ox)

print("\nRandom-input export check")
print("-------------------------")
print(f"Max absolute difference : {random_diff.max():.10f}")
print(f"Mean absolute difference: {random_diff.mean():.10f}")


ONNX Runtime available providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
ONNX Runtime session providers: ['CUDAExecutionProvider', 'CPUExecutionProvider']
ONNX input: input ['batch_size', 1, 8, 8] tensor(float)
ONNX output: output ['batch_size', 1] tensor(float)

PyTorch vs ONNX numerical comparison
-------------------------------------
Max absolute difference : 0.0001161695
Mean absolute difference: 0.0000562161
RMSE                    : 0.0000681368

Starting position: PyTorch vs ONNX
----------------------------------
Move    UCI         PyTorch         ONNX       |diff|
-------------------------------------------------------
d4      d2d4       0.522132     0.522227   0.00009513
Nf3     g1f3       0.519819     0.519930   0.00011152
e4      e2e4       0.512114     0.512166   0.00005221
c4      c2c4       0.510700     0.510731   0.00003171
Nc3     b1c3       0.501796     0.501680   0.00011617
e3      e2e3       0.499788     0.499761   0.00002721